# Finetuning

In [21]:
 !pip install openai

Prompting

In [22]:
import os
from openai import OpenAI

In [ ]:
#set the api key
openai_api_key='your-api-key'
os.environ["OPENAI_API_KEY"]=openai_api_key
client=OpenAI(api_key=openai_api_key)
def prompt_based_query(user_query):
  response=client.chat.completions.create(
      model="gpt-4.1-mini",
      messages=[{"role":"system","content":"you are a financial analyst"},{"role":"user","content":f"Answer this query:{user_query}"}
         ],
      temperature=0.2
      )
  return response.choices[0].message.content
print(prompt_based_query('explain the revenue growth in simple terms'))

Sure! Revenue growth means that a company is making more money from selling its products or services compared to before. It shows that the business is expanding and attracting more customers or selling more to existing customers. Simply put, if a company’s revenue is growing, it means it’s earning more income over time.


## RAG

Load Document

In [24]:
!pip install langchian faiss-cpu openai langchain_community tiktoken

In [25]:
#step 1 load document
from langchain_community.document_loaders import TextLoader
loader=TextLoader('/content/company_data.txt')
documents=loader.load()

In [ ]:
#step 2 create the embedding + vector DB
import os
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
embeddings=OpenAIEmbeddings(openai_api_key=os.environ.get('your-api-key'))
vector_db=FAISS.from_documents(documents,embeddings)

In [27]:
#step 3 Retrieval + Generation
def rag_query(query):
  docs=vector_db.similarity_search(query,k=3)
  context=" ".join(doc.page_content for doc in docs)
  response=client.chat.completions.create(
       model="gpt-4.1-mini",
       messages=[
           {"role":"system","content":"use the provided context to answer"},
           {"role":"user","content": f"context:{context} \n\nQuestions:{query}"}
       ]
       )
  return response.choices[0].message.content

In [28]:
print(rag_query("what is our company refund policy"))

ABC Retail's refund policy allows customers to request a refund within 7 days of delivery. Once the refund is approved, it is processed within 5-7 business days.


# Fine Tuning(PEFT +LoRA)

In [29]:
!pip install transformers datasets peft accelerate bitsandbytes

In [30]:
#step 1 : Load model
from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig
import bitsandbytes as bnb
model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer=AutoTokenizer.from_pretrained(model_name)
#define the bitsandbytes config for 8 bit quantization
quantization_config=BitsAndBytesConfig(load_in_8bit=True)
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [31]:
#step 2 : Apply LoRA(PEFT)
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training
#preapre the model for k bit trining
model = prepare_model_for_kbit_training(model)
lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.5,
    bias="none",
    task_type="CAUSAL_LM"
)
model=get_peft_model(model,lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [32]:
#step 3: dataset preperation
from datasets import Dataset
data=[
    {"text":"Q: what is ROI?\nA: Return on Investment is a probability metric"},
    {"text":"Q: Define Churn?\nA: Percentage of customer leaving the services"}
]
dataset=Dataset.from_list(data)

In [33]:
#step 4: tokenization
def tokenize_function(example):
  return tokenizer(example["text"],truncation=True,padding="max_length")
tokenized_dataset=dataset.map(tokenize_function)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [34]:
#step 5: Training
from transformers import Trainer , TrainingArguments
#add labels to the tokenized_dataset for causal language mdelling
def add_labels_to_dataset(examples):
  examples['labels']=examples['input_ids']
  return examples
tokenized_dataset=tokenized_dataset.map(add_labels_to_dataset,batched=True)
training_args=TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=50
)
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)
trainer.train()

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


TrainOutput(global_step=2, training_loss=5.210411071777344, metrics={'train_runtime': 18.1936, 'train_samples_per_second': 0.22, 'train_steps_per_second': 0.11, 'total_flos': 50903717511168.0, 'train_loss': 5.210411071777344, 'epoch': 2.0})

In [35]:
#step 6: inferences
def generate_response(prompt):
  inputs=tokenizer(prompt,return_tensors="pt").to("cuda")
  outputs=model.generate(**inputs,max_new_tokens=100)
  return tokenizer.decode(outputs[0])

In [36]:
print(generate_response("what is ROI"))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype

<s> what is ROI and the same time, and the same time to the same time to the same time to the world.















































































